# Clase 30: Despliegue básico de modelos

En esta clase vamos a convertir un modelo entrenado en un servicio real que puede recibir solicitudes y devolver predicciones.

> **Idea central:** Desplegar un modelo significa convertirlo en un servicio que otros sistemas o personas pueden usar directamente, sin abrir un notebook.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import pickle
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
print('Librerías cargadas')
print()
print('Pipeline de despliegue:')
print('1. ENTRENAR  →  2. GUARDAR  →  3. CARGAR  →  4. PREDECIR  →  5. SERVIR')

## 1. Entrenar el modelo

Primero entrenamos el modelo como en cualquier clase. Al final de este notebook lo vamos a convertir en un servicio real.

In [ ]:
# Crear dataset
n = 400
cursos = np.random.choice(['A', 'B', 'C'], size=n, p=[0.4, 0.4, 0.2])
nota_mate = np.where(cursos == 'A', np.random.normal(7.5, 1.2, n),
    np.where(cursos == 'B', np.random.normal(7.0, 1.3, n), np.random.normal(5.8, 1.5, n))).clip(1, 10)
nota_lengua = np.where(cursos == 'A', np.random.normal(7.2, 1.1, n),
    np.where(cursos == 'B', np.random.normal(6.8, 1.2, n), np.random.normal(5.5, 1.4, n))).clip(1, 10)
asistencia = np.where(cursos == 'C', np.random.uniform(0.55, 0.85, n), np.random.uniform(0.70, 0.99, n))
edades = np.random.randint(18, 35, size=n)
prob = (nota_mate * 0.4 + nota_lengua * 0.4 + asistencia * 10 * 0.2) / 10
aprobado = (np.random.uniform(0, 1, n) < prob).astype(int)

FEATURES = ['nota_matematicas', 'nota_lengua', 'asistencia', 'edad']
df = pd.DataFrame({'nota_matematicas': nota_mate.round(1), 'nota_lengua': nota_lengua.round(1),
                   'asistencia': asistencia.round(2), 'edad': edades, 'aprobado': aprobado})

X = df[FEATURES]
y = df['aprobado']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalizar
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Entrenar
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train_sc, y_train)

acc = accuracy_score(y_test, modelo.predict(X_test_sc))
print(f'Modelo entrenado. Precisión en test: {acc:.2%}')
print(f'Dataset: {len(df)} estudiantes, {len(FEATURES)} variables')

## 2. Guardar el modelo con joblib

El modelo existe ahora solo en memoria. Cuando cerremos el notebook, desaparece. Para que persista (y para usarlo desde otros programas), debemos guardarlo en disco.

In [ ]:
# Guardar el modelo Y el scaler
# IMPORTANTE: si olvidas el scaler, la API no va a funcionar correctamente
joblib.dump(modelo, 'modelo_estudiantes.pkl')
joblib.dump(scaler, 'scaler_estudiantes.pkl')

print('Archivos guardados:')
for archivo in ['modelo_estudiantes.pkl', 'scaler_estudiantes.pkl']:
    tamano = os.path.getsize(archivo) / 1024
    print(f'  {archivo}: {tamano:.1f} KB')

print()
print('Estos archivos contienen todo lo necesario para hacer predicciones:')
print('- modelo_estudiantes.pkl: los árboles entrenados del Random Forest')
print('- scaler_estudiantes.pkl: la media y desviación estándar del training set')

## 3. Cargar el modelo (simulamos "otro programa")

Ahora borramos las variables de memoria para simular que somos un programa nuevo que solo tiene acceso a los archivos guardados.

In [ ]:
# Borrar del contexto para simular un programa nuevo
del modelo, scaler
print('Modelo y scaler borrados de la memoria')
print()

# Cargar desde los archivos
modelo_cargado = joblib.load('modelo_estudiantes.pkl')
scaler_cargado = joblib.load('scaler_estudiantes.pkl')

print('Modelo cargado desde disco!')
print(f'Tipo: {type(modelo_cargado).__name__}')
print(f'Número de árboles: {modelo_cargado.n_estimators}')

# Verificar que funciona igual
acc_nuevo = accuracy_score(y_test, modelo_cargado.predict(scaler_cargado.transform(X_test)))
print(f'Precisión del modelo cargado: {acc_nuevo:.2%}')
print('¡Funciona exactamente igual que antes de guardarlo!')

## 4. Función de predicción limpia

Antes de hacer la API, creamos una función que encapsule toda la lógica de predicción. Esto hace el código más fácil de mantener y probar.

In [ ]:
def predecir_estudiante(nota_matematicas, nota_lengua, asistencia, edad):
    """
    Predice si un estudiante aprobará.
    
    Retorna un diccionario con la predicción completa.
    """
    # Preparar los datos como array
    X = np.array([[nota_matematicas, nota_lengua, asistencia, edad]])
    
    # Aplicar la MISMA normalización que se usó al entrenar
    X_scaled = scaler_cargado.transform(X)
    
    # Predecir
    prediccion = int(modelo_cargado.predict(X_scaled)[0])
    probabilidad = float(modelo_cargado.predict_proba(X_scaled)[0][1])
    
    # Determinar nivel de confianza
    if probabilidad > 0.80 or probabilidad < 0.20:
        confianza = 'alta'
    elif probabilidad > 0.60 or probabilidad < 0.40:
        confianza = 'media'
    else:
        confianza = 'baja'
    
    return {
        'aprobado': bool(prediccion),
        'probabilidad': round(probabilidad, 3),
        'nivel_confianza': confianza,
        'mensaje': f'Probabilidad de aprobar: {probabilidad:.1%} (confianza {confianza})'
    }

# Probar con distintos perfiles de estudiante
print('=== Pruebas de la función de predicción ===')
casos = [
    (8.5, 7.8, 0.95, 22, 'Estudiante destacado'),
    (6.0, 6.2, 0.80, 25, 'Estudiante promedio'),
    (4.5, 4.1, 0.62, 30, 'Estudiante en riesgo'),
]

for nota_m, nota_l, asist, edad, descripcion in casos:
    resultado = predecir_estudiante(nota_m, nota_l, asist, edad)
    estado = 'APROBADO' if resultado['aprobado'] else 'REPROBADO'
    print(f'{descripcion}:')
    print(f'  → {estado} | {resultado["probabilidad"]:.1%} | Confianza: {resultado["nivel_confianza"]}')
    print()

## 5. Crear la API con Flask

Ahora creamos el archivo `app.py` — el servidor que otros programas pueden consultar. Este archivo se ejecuta en una terminal separada.

In [ ]:
# Creamos el archivo app.py con el contenido de la API
app_code = '''
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)

# Cargar modelo y scaler al iniciar el servidor (una sola vez)
modelo = joblib.load('modelo_estudiantes.pkl')
scaler = joblib.load('scaler_estudiantes.pkl')
FEATURES = ['nota_matematicas', 'nota_lengua', 'asistencia', 'edad']

print('Servidor iniciado. Modelo cargado correctamente.')


@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'ok',
        'modelo': 'RandomForest Estudiantes',
        'version': '1.0'
    })


@app.route('/predict', methods=['POST'])
def predict():
    try:
        datos = request.get_json()
        
        # Validar campos requeridos
        for campo in FEATURES:
            if campo not in datos:
                return jsonify({'error': f'Campo faltante: {campo}'}), 400
        
        # Preparar datos para el modelo
        X = np.array([[datos[f] for f in FEATURES]])
        X_scaled = scaler.transform(X)
        
        # Predecir
        prediccion = int(modelo.predict(X_scaled)[0])
        probabilidad = float(modelo.predict_proba(X_scaled)[0][1])
        
        # Nivel de confianza
        if probabilidad > 0.80 or probabilidad < 0.20:
            confianza = 'alta'
        elif probabilidad > 0.60 or probabilidad < 0.40:
            confianza = 'media'
        else:
            confianza = 'baja'
        
        return jsonify({
            'aprobado': bool(prediccion),
            'probabilidad': round(probabilidad, 3),
            'nivel_confianza': confianza,
            'mensaje': 'El estudiante aprobará' if prediccion else 'El estudiante está en riesgo'
        })
    
    except Exception as e:
        return jsonify({'error': str(e)}), 500


if __name__ == '__main__':
    app.run(debug=True, port=5000)
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print('Archivo app.py creado!')
print()
print('Para correr el servidor:')
print('  1. Abre una terminal')
print('  2. Navega a esta carpeta')
print('  3. Ejecuta: python app.py')
print('  4. El servidor quedará en: http://localhost:5000')

## 6. Probar la API con requests

Con el servidor corriendo en otra terminal (`python app.py`), ejecuta las siguientes celdas para probar la API.

In [ ]:
# NOTA: Esta celda requiere que el servidor esté corriendo en otra terminal
# Si no está corriendo, obtendrás un error de conexión

try:
    import requests
    BASE_URL = 'http://localhost:5000'
    
    # a) Health check
    resp = requests.get(f'{BASE_URL}/health')
    print('Health check:', resp.json())
    print()
    
    # b) Predicción
    datos_estudiante = {
        'nota_matematicas': 7.5,
        'nota_lengua': 6.8,
        'asistencia': 0.90,
        'edad': 23
    }
    resp = requests.post(f'{BASE_URL}/predict', json=datos_estudiante)
    print('Predicción:', resp.json())

except Exception as e:
    print('El servidor no está corriendo.')
    print('Para probarlo: abre una terminal y ejecuta "python app.py"')
    print(f'Error: {e}')

## 7. Probar varios estudiantes a la vez

In [ ]:
try:
    import requests
    BASE_URL = 'http://localhost:5000'
    
    estudiantes = [
        {'nota_matematicas': 8.5, 'nota_lengua': 7.8, 'asistencia': 0.95, 'edad': 22},
        {'nota_matematicas': 6.0, 'nota_lengua': 6.2, 'asistencia': 0.80, 'edad': 25},
        {'nota_matematicas': 4.5, 'nota_lengua': 4.1, 'asistencia': 0.62, 'edad': 30},
        {'nota_matematicas': 9.0, 'nota_lengua': 8.5, 'asistencia': 0.98, 'edad': 20},
    ]
    
    print(f'{"Notas":>12} {"Asist":>8} {"Resultado":>15} {"Prob":>8} {"Confianza":>12}')
    print('-' * 60)
    
    for est in estudiantes:
        resp = requests.post(f'{BASE_URL}/predict', json=est)
        r = resp.json()
        resultado = 'APROBADO' if r['aprobado'] else 'REPROBADO'
        notas = f"{est['nota_matematicas']}/{est['nota_lengua']}"
        print(f'{notas:>12} {est["asistencia"]:>8.0%} {resultado:>15} {r["probabilidad"]:>8.1%} {r["nivel_confianza"]:>12}')

except Exception as e:
    print('El servidor no está corriendo. Ver celda anterior para instrucciones.')

## 8. Manejo de errores de la API

In [ ]:
try:
    import requests
    BASE_URL = 'http://localhost:5000'
    
    print('=== Pruebas de manejo de errores ===')
    print()
    
    # Campo faltante
    resp = requests.post(f'{BASE_URL}/predict', json={'nota_matematicas': 7.5})
    print(f'Campo faltante → HTTP {resp.status_code}: {resp.json()}')
    
    # Datos vacíos
    resp = requests.post(f'{BASE_URL}/predict', json={})
    print(f'Sin datos → HTTP {resp.status_code}: {resp.json()}')

except Exception as e:
    print('El servidor no está corriendo. Ver instrucciones anteriores.')

## 9. El mundo real: lo que viene después

Lo que construimos hoy es un prototipo funcional. En producción, hay más capas:

In [ ]:
pipeline_produccion = {
    'Paso 1 - Lo que hicimos hoy': [
        'Entrenar modelo con scikit-learn',
        'Guardar con joblib',
        'API básica con Flask',
    ],
    'Paso 2 - Mejoras inmediatas': [
        'FastAPI en lugar de Flask (más moderno, con documentación automática)',
        'Streamlit para una interfaz visual rápida',
        'Validación robusta de datos de entrada',
    ],
    'Paso 3 - Para producción real': [
        'Docker: empaquetar app + dependencias en contenedor',
        'Cloud (AWS/GCP/Azure): correr 24/7 y escalar',
        'Logging: registrar cada predicción',
        'Monitoring: detectar cuando el modelo empeora',
        'CI/CD: automatizar testing y despliegue',
    ]
}

for etapa, pasos in pipeline_produccion.items():
    print(f'\n{etapa}:')
    for paso in pasos:
        print(f'  ✓ {paso}')

print()
print('En este bootcamp llegamos hasta el Paso 1.')
print('El Paso 2 es alcanzable en pocas semanas de práctica adicional.')
print('El Paso 3 es la diferencia entre Data Scientist y ML Engineer.')

## Resumen

### Lo que aprendimos hoy:

1. **joblib.dump / joblib.load** → guardar y cargar modelos entrenados
2. **Siempre guarda también el scaler** → si normalizaste al entrenar, necesitas el mismo scaler en producción
3. **Función de predicción limpia** → encapsula la lógica antes de ponerla en una API
4. **Flask** → framework web para crear endpoints HTTP en Python
5. **POST /predict** → recibe JSON con datos, devuelve JSON con predicción
6. **requests** → para probar la API desde Python

### El concepto más importante:

> Un modelo que solo funciona en un notebook no es un producto. El despliegue es lo que convierte el trabajo del científico de datos en valor real para el mundo.